# Hy3 API - 非流式 vs 流式对比

本示例对比非流式（同步）和流式（异步）两种请求方式的性能差异，重点关注首 Token 时延（TTFT）和总耗时。

In [ ]:
from openai import OpenAI
import time
import os
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(
    api_key=os.getenv("API_KEY", "EMPTY"),
    base_url=os.getenv("BASE_URL", "http://127.0.0.1:8000/v1")
)

## 性能指标

- **TTFT** (Time To First Token): 从请求开始到收到第一个 token 的时间
- **Total Time**: 从请求开始到收到完整响应的时间
- **Content Length**: 响应内容的字符长度

In [ ]:
test_message = "请详细解释什么是人工智能大模型，包括其工作原理、主要特点和应用场景"

## 非流式请求

In [ ]:
print("=== 非流式请求 ===")
start_time = time.time()

response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": test_message}],
    temperature=0.9,
    top_p=1.0,
)

total_time = time.time() - start_time
content = response.choices[0].message.content

non_stream_result = {
    "total_time": total_time,
    "first_token_time": total_time,
    "content_length": len(content),
    "content": content,
}

print(f"总耗时: {total_time:.4f}s")
print(f"内容长度: {len(content)} 字符")
print(f"回复(截断): {content[:100]}...")

## 流式请求

In [ ]:
print("\n=== 流式请求 ===")
start_time = time.time()
first_token_time = None

stream = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": test_message}],
    temperature=0.9,
    top_p=1.0,
    stream=True,
)

full_content = ""
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        if first_token_time is None:
            first_token_time = time.time() - start_time
        full_content += chunk.choices[0].delta.content

total_time = time.time() - start_time

stream_result = {
    "total_time": total_time,
    "first_token_time": first_token_time or total_time,
    "content_length": len(full_content),
    "content": full_content,
}

print(f"首 Token 时延: {first_token_time:.4f}s")
print(f"总耗时: {total_time:.4f}s")
print(f"内容长度: {len(full_content)} 字符")

## 对比结果

In [ ]:
print("=" * 60)
print("【性能对比结果】")
print("-" * 60)
print(f"{'指标':<25} {'非流式':<15} {'流式':<15}")
print("-" * 60)
print(f"{'首 Token 时延 (TTFT)':<25} {non_stream_result['first_token_time']:.4f}s {stream_result['first_token_time']:.4f}s")
print(f"{'总耗时 (Total Time)':<25} {non_stream_result['total_time']:.4f}s {stream_result['total_time']:.4f}s")
print(f"{'内容长度':<25} {non_stream_result['content_length']:<15} {stream_result['content_length']:<15}")
print("-" * 60)

ttft_improvement = ((non_stream_result['first_token_time'] - stream_result['first_token_time']) / non_stream_result['first_token_time']) * 100
print(f"\n首 Token 时延提升: {ttft_improvement:.1f}%")

## 关键结论

1. **TTFT 优势**：流式请求的首 Token 时延通常比非流式低 80-90%
2. **总耗时相近**：两种方式的总耗时差异不大
3. **用户体验**：流式能让用户更早看到内容，减少等待焦虑
4. **适用场景**：
   - **流式**：实时对话、聊天界面、需要快速响应的场景
   - **非流式**：批量处理、后台任务、需要完整响应的场景